In [ ]:
import sys
from pathlib import Path

# Resolve project root directory to enable imports from src/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import pandas as pd
import yaml

from src.Parser.NeuralRingerTrainerConfiguration import NeuralRingerTrainerConfiguration
from src.core.Trainers.NeuralRingerTrainer import NeuralRingerTrainer
from src.core.Datasets.SplitManifest import SplitManifest
from src.core.Validation.ResultAggregator import ResultAggregator
from src.core.Validation.HoldoutEvaluator import HoldoutEvaluator, HoldoutEvaluationResult
from src.core.Validation.EfficiencyPlotter import EfficiencyPlotter
from src.core.Validation.ModelValidator import ModelValidator

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

### Load Configuration & Configure Paths

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "config/NeuralRinger/ModelV6_HighBatch_newExtraction_20_regions_100RingsNoFilter.yaml"
RESULTS_PATH = PROJECT_ROOT / "results"
DATA_PATH = PROJECT_ROOT / "data/consolidated"
OUTPUT_DIR = PROJECT_ROOT / "Plots/Validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "r") as f:
    config_dict = yaml.safe_load(f)

# Override dataset drive_path with local data directory if available
if DATA_PATH.exists():
    config_dict["dataset"]["parameters"]["drive_path"] = str(DATA_PATH)

config = NeuralRingerTrainerConfiguration.model_validate(config_dict)
print(f"Loaded experiment: {config.config_name}")
print(f"Architecture: {config.model.object_name}, Input dimension: {config.model.parameters.get('input_dim')}")

### Step-by-Step Data Loading & Model Creation

Directly load dataset partitions, inspect feature matrices, and instantiate the PyTorch neural network.

In [ ]:
trainer = NeuralRingerTrainer(config)
dataset = trainer.full_dataset
model = trainer.factory.create_model()
device = trainer.device

print(f"Target computation device: {device}")
print(f"Model instance: {model.__class__.__name__}")
print(f"Discovered dataset region files: {len(dataset)}")

if len(dataset) > 0:
    features, labels, file_path = dataset[0]
    print(f"\nInspecting region file: {file_path}")
    print(f"Feature matrix shape: {features.shape}")
    print(f"Target vector shape:  {labels.shape}")
    print(f"Signal events (class 1):     {(labels == 1).sum()}")
    print(f"Background events (class 0): {(labels == 0).sum()}")
    print(f"Active ring column indices:  {len(dataset.ring_column_indices)}")